In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from src.mbe import patch_mbe 
import torch 
from src.gapt import GatedPhaseTransition

# ----- load model -----
model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# ----- initialize GAPT ----- 
gapt = GatedPhaseTransition(
    p_m=200, # patience plateaued ce loss 
    p_a=20 # patience for plateaued mbe loss 
)

# ----- compute loss ----
num_layers = model.config.num_hidden_layers
per_layer_mbe_mask = torch.ones(num_layers) # <- require ablation on this mask
patch_size = 3

inputs = tokenizer("Hello, how are you?", return_tensors="pt")
assert len(inputs["input_ids"][0]) % patch_size == 0, "patch size needs to divide seq len here"

def compute_loss(model, inputs, patch_size: int, per_layer_mbe_mask: torch.Tensor, gapt: GatedPhaseTransition):
    outputs = model(**inputs, labels=inputs["input_ids"], output_hidden_states=True, return_dict=True)
    ce_loss = outputs.loss 

    mbe_per_layer = torch.stack([patch_mbe(h, patch_size).float() for h in outputs.hidden_states[1:]])
    mbe_loss = (mbe_per_layer * per_layer_mbe_mask).mean()

    loss = gapt.step(ce_loss, mbe_loss)
    return loss